# Chapter 10 — SOLUTIONS: Introduction to Reinforcement Learning

**Instructor file — share only after exercise session.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
np.random.seed(42)

# Grid environment setup
GRID = np.array([[3,0,0,0],[0,1,0,1],[0,0,0,1],[1,0,0,2]])
ROWS, COLS, N_STATES, N_ACTIONS = 4, 4, 16, 4
MOVE_DR = {0:(0,-1), 1:(1,0), 2:(0,1), 3:(-1,0)}

REWARDS = {0: -0.1, 1: -1.0, 2: +1.0, 3: -0.1}

_state = [0]

def env_reset():
    _state[0] = 0
    return 0

def env_step(action):
    r, c = _state[0] // COLS, _state[0] % COLS
    dr, dc = MOVE_DR[action]
    nr, nc = max(0, min(ROWS-1, r+dr)), max(0, min(COLS-1, c+dc))
    ns = nr * COLS + nc
    cell_type = GRID[nr, nc]
    reward = REWARDS[cell_type]
    done = cell_type in [1, 2]
    _state[0] = ns
    return ns, reward, done

def run_episodes(n=1000, max_steps=100, collect_rewards=False):
    successes, total_rewards = 0, []
    for _ in range(n):
        state = env_reset()
        total_r = 0
        for _ in range(max_steps):
            action = np.random.randint(N_ACTIONS)
            ns, r, done = env_step(action)
            total_r += r
            state = ns
            if done:
                if r > 0:
                    successes += 1
                break
        total_rewards.append(total_r)
    if collect_rewards:
        return successes / n, np.mean(total_rewards), total_rewards
    return successes / n, np.mean(total_rewards)

print('Setup complete!')

## Task 1 Solution: Baseline — Random Agent

In [ ]:
# Baseline with default rewards
REWARDS = {0: -0.1, 1: -1.0, 2: +1.0, 3: -0.1}
success_rate, avg_reward = run_episodes(1000)

print('=== Baseline (default rewards) ===')
print(f'  Hole reward:      {REWARDS[1]}')
print(f'  Goal reward:      {REWARDS[2]}')
print(f'  Step penalty:     {REWARDS[0]}')
print(f'  Success rate:     {success_rate:.1%}')
print(f'  Average reward:   {avg_reward:.3f}')

## Task 2 Solution: Reward Shaping — Reduce the Hole Penalty

In [ ]:
# 2a: Reduce hole penalty from -1.0 to -0.1
REWARDS = {0: -0.1, 1: -0.1, 2: +1.0, 3: -0.1}
success_2, avg_2 = run_episodes(1000)

print('=== Reduced hole penalty (hole = -0.1) ===')
print(f'  Success rate:   {success_2:.1%}  (baseline: ~same)')
print(f'  Average reward: {avg_2:.3f}  (baseline was lower — less punishment for falling)')
print()

# 2b: Discussion
print('Discussion:')
print('• For a RANDOM agent: success rate barely changes.')
print('  The agent has no policy — it cannot reason about rewards.')
print('  It still falls into holes at the same frequency.')
print()
print('• Average reward goes UP because holes are now less costly.')
print()
print('• For an OPTIMAL/LEARNED agent: behavior would change significantly!')
print('  With mild hole penalties, the agent might take shorter but riskier paths.')
print('  This is reward shaping: changing rewards changes the optimal strategy.')

## Task 3 Solution: Reward Shaping — Increase the Step Penalty

In [ ]:
# 3a & 3b: Higher step penalty
REWARDS = {0: -0.5, 1: -1.0, 2: +1.0, 3: -0.5}
success_3, avg_3 = run_episodes(1000)

print('=== Higher step penalty (free = -0.5) ===')
print(f'  Success rate:   {success_3:.1%}')
print(f'  Average reward: {avg_3:.3f}  (much lower — each step costs more)')
print()

# 3c: Discussion
print('Discussion:')
print('• For a RANDOM agent: success rate does not improve.')
print('  Reward shaping cannot make a random agent smarter —')
print('  the agent takes the same random actions regardless of the step cost.')
print()
print('• Average reward drops sharply — each random step is now costly.')
print()
print('• Reward shaping is powerful ONLY when combined with a LEARNING agent.')
print('  A high step penalty tells a Q-Learning agent: reach the goal FAST.')
print('  It has no effect on a pure random agent.')
print()
print('→ This is why reward design is one of the hardest parts of RL!')

## Bonus Solution: Visualize the Return Distribution

In [ ]:
REWARDS = {0: -0.1, 1: -1.0, 2: +1.0, 3: -0.1}  # reset to default

_, _, episode_rewards = run_episodes(500, collect_rewards=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(episode_rewards, bins=40, color='#3498db', alpha=0.75, edgecolor='white')
ax.axvline(np.mean(episode_rewards), color='red', linestyle='--',
           linewidth=2, label=f'Mean = {np.mean(episode_rewards):.2f}')
ax.set_xlabel('Total Episode Return', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Episode Returns — Random Agent (500 episodes)', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Mean return:   {np.mean(episode_rewards):.3f}')
print(f'Min return:    {np.min(episode_rewards):.3f}  (long episodes with many steps)')
print(f'Max return:    {np.max(episode_rewards):.3f}  (reached goal quickly)')
print()
print('The distribution is left-skewed: most episodes are long and accumulate step penalties.')
print('A learned agent would shift this distribution to the right (higher returns).')